# 📊 Module 7: Daily Sales Prediction (Random Forest Regressor)

In this notebook, we aggregate transaction data into daily revenue, construct temporal calendar features, lag features, and rolling mean features, train a **Random Forest Regressor**, evaluate performance metrics (**MAE, RMSE, R²**), and forecast future daily revenue.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print(f"Loaded Cleaned Data: {df.shape[0]:,} rows")

### Step 1: Aggregate Daily Sales

In [ ]:
df['Date'] = df['InvoiceDate'].dt.date
daily = df.groupby('Date')['TotalPrice'].sum().reset_index()
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values('Date').reset_index(drop=True)
daily.head()

### Step 2: Feature Engineering (Calendar, Lags, Rolling Averages)

In [ ]:
df_feat = daily.copy()
df_feat['DayOfWeek'] = df_feat['Date'].dt.dayofweek
df_feat['Month'] = df_feat['Date'].dt.month
df_feat['Day'] = df_feat['Date'].dt.day
df_feat['Quarter'] = df_feat['Date'].dt.quarter
df_feat['Is_Weekend'] = df_feat['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Lag Features
df_feat['lag_1'] = df_feat['TotalPrice'].shift(1)
df_feat['lag_7'] = df_feat['TotalPrice'].shift(7)
df_feat['lag_14'] = df_feat['TotalPrice'].shift(14)
df_feat['lag_30'] = df_feat['TotalPrice'].shift(30)

# Rolling Features
df_feat['rolling_7'] = df_feat['TotalPrice'].shift(1).rolling(7).mean()
df_feat['rolling_14'] = df_feat['TotalPrice'].shift(1).rolling(14).mean()

df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Feature Dataset Shape: {df_feat.shape}")
df_feat.head()

### Step 3: Train-Test Split & Random Forest Training

In [ ]:
feature_cols = ['DayOfWeek', 'Month', 'Day', 'Quarter', 'Is_Weekend', 
                'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_7', 'rolling_14']

X = df_feat[feature_cols]
y = df_feat['TotalPrice']

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=12)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"📊 Model Evaluation Results:")
print(f"  Mean Absolute Error (MAE): £{mae:,.2f}")
print(f"  Root Mean Sq Error (RMSE): £{rmse:,.2f}")
print(f"  R² Score (Accuracy): {r2:.4f}")

### Step 4: Actual vs Predicted Sales Plot

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test.values, label='Actual Daily Sales', color='#10B981', linewidth=2)
plt.plot(preds, label='Predicted Sales (RF)', color='#F59E0B', linestyle='--', linewidth=2)
plt.title('Daily Sales Forecasting: Actual vs Predicted')
plt.xlabel('Test Samples')
plt.ylabel('Revenue (£)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 5: Save Model to Disk

In [ ]:
model_path = os.path.join("..", "models", "sales_forecast_rf.joblib")
joblib.dump(rf, model_path)
print(f"Sales forecast model saved successfully to: {model_path}")